# EDA 1-4 for Project 02 Amazon Review Data

This notebook implements the EDA logic from `EDA.pdf` for the six local JSONL tables:

- User reviews:
  - `Beauty_and_Personal_Care.jsonl`
  - `Clothing_Shoes_and_Jewelry.jsonl`
  - `Sports_and_Outdoors.jsonl`
- Item metadata:
  - `meta_Beauty_and_Personal_Care.jsonl`
  - `meta_Clothing_Shoes_and_Jewelry.jsonl`
  - `meta_Sports_and_Outdoors.jsonl`

## Common Definitions

| Term | Meaning used in this notebook |
|---|---|
| `review_domain` / category | The source review file: `Beauty_and_Personal_Care`, `Clothing_Shoes_and_Jewelry`, or `Sports_and_Outdoors`. |
| Time window | Review rows where `timestamp` is from `2020-01-01 00:00:00` to `2022-12-31 23:59:59.999`. Timestamps are Unix milliseconds. |
| `asin` | The reviewed Amazon item ID. Often a child variant, like one size/color. |
| `parent_asin` | The product-family ID. This is usually the better recommender item key. |
| Metadata category | Fields like `main_category` or `categories` inside metadata. This notebook does not use metadata category as the first domain definition because it can be null or mixed. |

## 1. ASIN Integrity Audit

This is the "can we safely join and model this data?" check.

| Check | Exact meaning | Why it matters |
|---|---|---|
| Same `parent_asin` in 2+ review domains | Build a map like `parent_asin -> {Beauty, Clothing, Sports}` using review rows in 2020-2022. Flag any `parent_asin` whose domain set has size > 1. | If the same product family appears in multiple domains, we need to avoid double-counting it or treating it as cross-domain signal when it is actually the same item. |
| Same `asin` maps to 2+ `parent_asin` | Build `asin -> {parent_asin}` from reviews. Flag `asin` values with more than one parent. | This is suspicious. A child item should usually belong to one parent. If not, item identity is unstable. |
| Every review has usable `parent_asin` | Count review rows where `parent_asin` is missing, empty, or null. | If many rows lack `parent_asin`, we cannot use parent-level recommendation cleanly. |
| Review `parent_asin` exists in metadata | For each domain, check whether review `parent_asin` exists in the matching metadata file. | Needed for title/features/description/images joins. Missing metadata means no text/image features for that item. |
| Metadata duplicate `parent_asin` | Check if the same metadata file has duplicate rows for one `parent_asin`. | Duplicate item metadata can create repeated or conflicting feature rows. |

## 2. Dataset Shape And Sparsity

This tells us whether the recommender problem is healthy or too sparse.

| Check | Exact meaning | Why it matters |
|---|---|---|
| Row counts by domain | Number of review rows in Beauty, Clothing, Sports, both all-time and 2020-2022. | Shows usable data size. |
| Unique users | Count distinct `user_id` per domain and across all domains. | Recommendation depends on repeated user behavior. |
| Unique items | Count unique `asin` and unique `parent_asin`. | Decides whether item-level or parent-level modeling is better. |
| Interactions per user | For each user, count review rows. Summarize mean, median, p90, p99. | If most users have 1 review, sequential/personalized modeling is harder. |
| Interactions per item | For each `parent_asin`, count reviews. Summarize distribution. | Finds cold-start items and popular items. |
| Rating distribution | Count ratings 1, 2, 3, 4, 5 by domain. | Helps decide whether to treat reviews as implicit feedback or rating prediction. |

For this project, this matters because if Beauty has many users with only one Beauty review, Clothing/Sports histories may be useful as auxiliary user preference signals.

## 3. Cross-Domain Transfer Viability

This checks whether Clothing/Sports can actually help Beauty recommendation.

| Check | Exact meaning | Why it matters |
|---|---|---|
| Shared users across domains | Count users who appear in Beauty and Clothing, Beauty and Sports, or all three. | Cross-domain recommendation only works if users overlap across domains. |
| Source history before Beauty event | For each Beauty review in 2020-2022, check whether the same user had earlier Clothing/Sports reviews. | This avoids using future information to predict past Beauty behavior. |
| User sequence length | Count how many prior interactions each Beauty user has in other domains. | More prior behavior means stronger transfer signal. |
| Cross-domain rating tendency | Compare a user's average Clothing/Sports ratings to their Beauty ratings. | Tests whether user preference style transfers. |
| Category path similarity | Use metadata `categories` to see what kinds of Clothing/Sports products overlap with Beauty users. | Helps explain transfer: style, self-care, fitness, outdoor lifestyle, etc. |

## 4. Metadata/Text/Image Readiness

This checks whether the side features are usable for embeddings.

| Check | Exact meaning | Why it matters |
|---|---|---|
| Missing title/features/description | For each metadata file, count empty `title`, `features`, `description`. | Text embeddings need meaningful text. |
| Text length distribution | Measure character/token length of title + features + description. | Helps choose truncation rules before BGE embedding. |
| Image coverage | Count metadata rows with at least one image URL, especially usable `large` or `hi_res`. | Needed for CLIP image embeddings. |
| Review image coverage | Count review rows with user-uploaded images. | Could be useful, but usually much sparser than metadata images. |
| Price availability | Count missing `price`. | Price may be useful, but if too sparse it should be optional. |
| Metadata quality by domain | Compare Beauty vs Clothing vs Sports coverage. | If one domain has much weaker metadata, embedding quality may differ. |

In [1]:
from __future__ import annotations

import csv
import gc
import json
import math
import statistics
import time
from bisect import bisect_left
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = print

PROJECT_ROOT = Path('/Users/frankwang1224/Projects/rcd_sys_proj02')
DATASET_DIR = PROJECT_ROOT / 'dataset'
OUTPUT_DIR = PROJECT_ROOT / 'eda_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

REVIEW_FILES = {
    'Beauty_and_Personal_Care': DATASET_DIR / 'Beauty_and_Personal_Care.jsonl',
    'Clothing_Shoes_and_Jewelry': DATASET_DIR / 'Clothing_Shoes_and_Jewelry.jsonl',
    'Sports_and_Outdoors': DATASET_DIR / 'Sports_and_Outdoors.jsonl',
}

META_FILES = {
    'Beauty_and_Personal_Care': DATASET_DIR / 'meta_Beauty_and_Personal_Care.jsonl',
    'Clothing_Shoes_and_Jewelry': DATASET_DIR / 'meta_Clothing_Shoes_and_Jewelry.jsonl',
    'Sports_and_Outdoors': DATASET_DIR / 'meta_Sports_and_Outdoors.jsonl',
}

START = datetime(2020, 1, 1, tzinfo=timezone.utc)
END = datetime(2022, 12, 31, 23, 59, 59, 999000, tzinfo=timezone.utc)
START_MS = int(START.timestamp() * 1000)
END_MS = int(END.timestamp() * 1000)

# Set these to an integer for quick smoke tests, e.g. 100_000.
# Keep as None for exact full-file EDA.
MAX_REVIEW_ROWS_PER_FILE = None
MAX_META_ROWS_PER_FILE = None

DOMAIN_BITS = {
    'Beauty_and_Personal_Care': 1,
    'Clothing_Shoes_and_Jewelry': 2,
    'Sports_and_Outdoors': 4,
}

DOMAIN_SHORT = {
    'Beauty_and_Personal_Care': 'Beauty',
    'Clothing_Shoes_and_Jewelry': 'Clothing',
    'Sports_and_Outdoors': 'Sports',
}

for label, files in [('review', REVIEW_FILES), ('metadata', META_FILES)]:
    for domain, path in files.items():
        if not path.exists():
            raise FileNotFoundError(f'Missing {label} file for {domain}: {path}')

print('Project root:', PROJECT_ROOT)
print('Output dir:', OUTPUT_DIR)
print('Review row cap per file:', MAX_REVIEW_ROWS_PER_FILE)
print('Metadata row cap per file:', MAX_META_ROWS_PER_FILE)
print('Date window:', START.isoformat(), 'to', END.isoformat())

Project root: /Users/frankwang1224/Projects/rcd_sys_proj02
Output dir: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs
Review row cap per file: None
Metadata row cap per file: None
Date window: 2020-01-01T00:00:00+00:00 to 2022-12-31T23:59:59.999000+00:00


In [2]:
def to_ms(ts):
    """Normalize timestamp values to Unix milliseconds."""
    if ts is None:
        return None
    try:
        ts = int(ts)
    except (TypeError, ValueError):
        return None
    digits = len(str(abs(ts)))
    if digits <= 11:
        return ts * 1000
    if digits >= 15:
        return ts // 1000
    return ts


def in_window(ts) -> bool:
    ms = to_ms(ts)
    return ms is not None and START_MS <= ms <= END_MS


def is_empty(value) -> bool:
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ''
    if isinstance(value, (list, tuple, dict, set)):
        return len(value) == 0
    return False


def safe_id(value):
    if value is None:
        return None
    value = str(value).strip()
    return value or None


def iter_jsonl(path: Path, max_rows=None):
    """Yield JSON objects from a JSONL file. Keeps memory flat."""
    bad_rows = 0
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            if max_rows is not None and i > max_rows:
                break
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                bad_rows += 1
                continue
    if bad_rows:
        print(f'[warn] {path.name}: skipped {bad_rows:,} malformed rows')


def decode_domains(mask: int) -> str:
    names = [DOMAIN_SHORT[d] for d, bit in DOMAIN_BITS.items() if mask & bit]
    return ', '.join(names)


def pct(n, d):
    return 0.0 if d == 0 else 100.0 * n / d


def distribution_summary(values):
    """Return count, mean, median, p90, p99, max for a list-like of counts."""
    values = list(values)
    if not values:
        return {'n': 0, 'mean': 0, 'median': 0, 'p90': 0, 'p99': 0, 'max': 0}
    values.sort()
    def q(q_value):
        idx = min(len(values) - 1, max(0, math.ceil(q_value * len(values)) - 1))
        return values[idx]
    return {
        'n': len(values),
        'mean': round(sum(values) / len(values), 3),
        'median': q(0.50),
        'p90': q(0.90),
        'p99': q(0.99),
        'max': values[-1],
    }


def text_values_to_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    return [text] if text else []


def combined_text_for_metadata(row):
    parts = []
    title = row.get('title')
    if not is_empty(title):
        parts.append(str(title).strip())
    parts.extend(text_values_to_list(row.get('features')))
    parts.extend(text_values_to_list(row.get('description')))
    return ' '.join(parts).strip()


def has_usable_metadata_image(row):
    images = row.get('images')
    if not isinstance(images, list) or not images:
        return False
    for image in images:
        if not isinstance(image, dict):
            continue
        if image.get('hi_res') or image.get('large') or image.get('thumb'):
            return True
    return False


def category_path(row):
    cats = row.get('categories')
    if isinstance(cats, list) and cats:
        return ' > '.join(str(x) for x in cats if str(x).strip())
    return None


def print_section(title: str, note: str | None = None):
    print('\n' + '=' * 100)
    print(title)
    print('=' * 100)
    if note:
        print(note)


def print_subsection(title: str, note: str | None = None):
    print('\n' + '-' * 100)
    print(title)
    print('-' * 100)
    if note:
        print(note)


def print_bullets(title: str, bullets):
    print_subsection(title)
    for bullet in bullets:
        print(f'- {bullet}')


def write_csv(path: Path, rows, fieldnames):
    rows = list(rows)
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    print(f'CSV saved: {path} ({len(rows):,} rows)')


def show_table(rows, max_rows=20, title: str | None = None, note: str | None = None):
    rows = list(rows)
    if title:
        print_subsection(title, note)
    elif note:
        print(note)
    if not rows:
        print('(no rows)')
        return
    shown_count = min(len(rows), max_rows)
    print(f'Showing {shown_count:,} of {len(rows):,} rows')
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        with pd.option_context(
            'display.max_columns', None,
            'display.width', 160,
            'display.max_colwidth', 80,
        ):
            display(df.head(max_rows))
        return
    except Exception:
        pass
    cols = list(rows[0].keys())
    shown = rows[:max_rows]
    widths = {
        c: min(48, max(len(c), *(len(str(r.get(c, ''))) for r in shown)))
        for c in cols
    }
    print(' | '.join(c.ljust(widths[c]) for c in cols))
    print('-|-'.join('-' * widths[c] for c in cols))
    for r in shown:
        values = []
        for c in cols:
            value = str(r.get(c, ''))
            if len(value) > widths[c]:
                value = value[: max(0, widths[c] - 3)] + '...'
            values.append(value.ljust(widths[c]))
        print(' | '.join(values))


def print_output_inventory(output_dir: Path):
    print_section('Output CSV inventory', 'These are the files produced by the notebook run.')
    files = sorted(output_dir.glob('*.csv'))
    if not files:
        print('(no CSV files found yet)')
        return
    for file in files:
        size_kb = file.stat().st_size / 1024
        print(f'- {file.name} ({size_kb:.1f} KB) -> {file}')

## Scan Metadata Files

This cell supports EDA 1 and EDA 4. It builds metadata parent sets for join checks and summarizes text/image/price readiness.

In [3]:
meta_parent_sets = {domain: set() for domain in META_FILES}
meta_parent_counts = {domain: Counter() for domain in META_FILES}
meta_summary = {}
meta_main_category_counts = {domain: Counter() for domain in META_FILES}
meta_first_category_counts = {domain: Counter() for domain in META_FILES}
meta_category_path_by_parent = defaultdict(Counter)
meta_parent_domain_mask = defaultdict(int)

for domain, path in META_FILES.items():
    t0 = time.time()
    stats = Counter()
    text_lengths = []
    print(f'\nScanning metadata: {domain} -> {path.name}')

    for row in iter_jsonl(path, MAX_META_ROWS_PER_FILE):
        stats['rows'] += 1
        parent = safe_id(row.get('parent_asin'))
        if not parent:
            stats['missing_parent_asin'] += 1
        else:
            meta_parent_sets[domain].add(parent)
            meta_parent_counts[domain][parent] += 1
            meta_parent_domain_mask[parent] |= DOMAIN_BITS[domain]

        for field in ['title', 'features', 'description', 'price', 'images', 'main_category', 'categories']:
            if is_empty(row.get(field)):
                stats[f'empty_{field}'] += 1

        combined = combined_text_for_metadata(row)
        if combined:
            stats['rows_with_embedding_text'] += 1
            text_lengths.append(len(combined))
        else:
            stats['empty_combined_text'] += 1

        if has_usable_metadata_image(row):
            stats['rows_with_usable_image'] += 1

        main_category = row.get('main_category')
        meta_main_category_counts[domain][main_category if main_category is not None else '(null)'] += 1

        cats = row.get('categories')
        if isinstance(cats, list) and cats:
            meta_first_category_counts[domain][str(cats[0])] += 1

        path_value = category_path(row)
        if parent and path_value:
            meta_category_path_by_parent[parent][path_value] += 1

    stats['scan_seconds'] = round(time.time() - t0, 1)
    stats.update({f'text_len_{k}': v for k, v in distribution_summary(text_lengths).items()})
    stats['duplicate_parent_asin_count'] = sum(1 for c in meta_parent_counts[domain].values() if c > 1)
    meta_summary[domain] = dict(stats)
    print(f"done {domain}: {stats['rows']:,} rows in {stats['scan_seconds']} sec")

meta_summary_rows = []
for domain, stats in meta_summary.items():
    rows = stats['rows']
    meta_summary_rows.append({
        'domain': domain,
        'metadata_rows': rows,
        'unique_parent_asin': len(meta_parent_sets[domain]),
        'duplicate_parent_asin': stats.get('duplicate_parent_asin_count', 0),
        'missing_parent_asin_pct': round(pct(stats.get('missing_parent_asin', 0), rows), 3),
        'empty_title_pct': round(pct(stats.get('empty_title', 0), rows), 3),
        'empty_features_pct': round(pct(stats.get('empty_features', 0), rows), 3),
        'empty_description_pct': round(pct(stats.get('empty_description', 0), rows), 3),
        'empty_price_pct': round(pct(stats.get('empty_price', 0), rows), 3),
        'usable_image_pct': round(pct(stats.get('rows_with_usable_image', 0), rows), 3),
        'embedding_text_pct': round(pct(stats.get('rows_with_embedding_text', 0), rows), 3),
        'text_len_median': stats.get('text_len_median', 0),
        'text_len_p90': stats.get('text_len_p90', 0),
        'text_len_p99': stats.get('text_len_p99', 0),
    })

show_table(
    meta_summary_rows,
    title='Metadata readiness summary by domain',
    note='This table supports EDA 1 and EDA 4. Percentages are based on metadata rows scanned.',
)
write_csv(OUTPUT_DIR / 'eda4_metadata_readiness_summary.csv', meta_summary_rows, list(meta_summary_rows[0].keys()))


Scanning metadata: Beauty_and_Personal_Care -> meta_Beauty_and_Personal_Care.jsonl
done Beauty_and_Personal_Care: 1,028,914 rows in 12.9 sec

Scanning metadata: Clothing_Shoes_and_Jewelry -> meta_Clothing_Shoes_and_Jewelry.jsonl
done Clothing_Shoes_and_Jewelry: 7,218,481 rows in 87.2 sec

Scanning metadata: Sports_and_Outdoors -> meta_Sports_and_Outdoors.jsonl
done Sports_and_Outdoors: 1,587,421 rows in 20.9 sec

----------------------------------------------------------------------------------------------------
Metadata readiness summary by domain
----------------------------------------------------------------------------------------------------
This table supports EDA 1 and EDA 4. Percentages are based on metadata rows scanned.
Showing 3 of 3 rows


,domain,metadata_rows,unique_parent_asin,duplicate_parent_asin,missing_parent_asin_pct,empty_title_pct,empty_features_pct,empty_description_pct,empty_price_pct,usable_image_pct,embedding_text_pct,text_len_median,text_len_p90,text_len_p99
0,Beauty_and_Personal_Care,1028914,1028914,0,0.0,0.005,27.537,45.149,63.002,99.992,99.999,627,2022,3312
1,Clothing_Shoes_and_Jewelry,7218481,7218481,0,0.0,0.008,10.317,47.448,79.546,99.861,99.999,541,1698,3476
2,Sports_and_Outdoors,1587421,1587421,0,0.0,0.007,16.919,34.837,69.426,99.966,99.999,591,1864,3448


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda4_metadata_readiness_summary.csv (3 rows)


## Scan Review Files

This cell supports EDA 1, 2, 3, and 4. It applies the 2020-2022 window, aggregates user/item sparsity, review image coverage, ASIN-parent mappings, and review-to-metadata coverage.

In [4]:
review_summary = {}
review_parent_sets = {domain: set() for domain in REVIEW_FILES}
review_parent_domain_mask = defaultdict(int)
asin_to_parents = defaultdict(set)
user_domain_mask = defaultdict(int)
all_users = set()
beauty_users = set()
beauty_user_rating_sum = defaultdict(float)
beauty_user_rating_count = defaultdict(int)
beauty_review_events_by_user = defaultdict(list)
review_user_counts_by_domain = {domain: Counter() for domain in REVIEW_FILES}
review_parent_counts_by_domain = {domain: Counter() for domain in REVIEW_FILES}
review_asin_sets_by_domain = {domain: set() for domain in REVIEW_FILES}
review_rating_counts_by_domain = {domain: Counter() for domain in REVIEW_FILES}
review_image_rows_by_domain = {domain: 0 for domain in REVIEW_FILES}
review_parent_missing_metadata_counts = {domain: 0 for domain in REVIEW_FILES}
review_parent_missing_metadata_examples = {domain: Counter() for domain in REVIEW_FILES}

for domain, path in REVIEW_FILES.items():
    t0 = time.time()
    stats = Counter()
    print(f'\nScanning reviews: {domain} -> {path.name}')

    for row in iter_jsonl(path, MAX_REVIEW_ROWS_PER_FILE):
        stats['rows_all_time'] += 1
        ts_ms = to_ms(row.get('timestamp'))
        window = ts_ms is not None and START_MS <= ts_ms <= END_MS
        if window:
            stats['rows_2020_2022'] += 1

        asin = safe_id(row.get('asin'))
        parent = safe_id(row.get('parent_asin'))
        user = safe_id(row.get('user_id'))
        rating = row.get('rating')

        if not parent:
            stats['missing_parent_asin_all_time'] += 1
            if window:
                stats['missing_parent_asin_2020_2022'] += 1
        if not asin:
            stats['missing_asin_all_time'] += 1
            if window:
                stats['missing_asin_2020_2022'] += 1
        if not user:
            stats['missing_user_id_all_time'] += 1
            if window:
                stats['missing_user_id_2020_2022'] += 1

        if is_empty(row.get('text')):
            stats['empty_text_all_time'] += 1
            if window:
                stats['empty_text_2020_2022'] += 1

        has_review_image = isinstance(row.get('images'), list) and len(row.get('images')) > 0
        if has_review_image:
            stats['review_rows_with_image_all_time'] += 1
            if window:
                stats['review_rows_with_image_2020_2022'] += 1
                review_image_rows_by_domain[domain] += 1

        if not window:
            continue

        if parent:
            review_parent_sets[domain].add(parent)
            review_parent_domain_mask[parent] |= DOMAIN_BITS[domain]
            review_parent_counts_by_domain[domain][parent] += 1
            if parent not in meta_parent_sets[domain]:
                review_parent_missing_metadata_counts[domain] += 1
                if len(review_parent_missing_metadata_examples[domain]) < 1000:
                    review_parent_missing_metadata_examples[domain][parent] += 1

        if asin:
            review_asin_sets_by_domain[domain].add(asin)
            if parent:
                asin_to_parents[asin].add(parent)

        if user:
            all_users.add(user)
            user_domain_mask[user] |= DOMAIN_BITS[domain]
            review_user_counts_by_domain[domain][user] += 1
            if domain == 'Beauty_and_Personal_Care':
                beauty_users.add(user)
                if ts_ms is not None:
                    beauty_review_events_by_user[user].append(ts_ms)
                if rating is not None:
                    try:
                        beauty_user_rating_sum[user] += float(rating)
                        beauty_user_rating_count[user] += 1
                    except (TypeError, ValueError):
                        pass

        if rating is not None:
            review_rating_counts_by_domain[domain][rating] += 1

    stats['scan_seconds'] = round(time.time() - t0, 1)
    review_summary[domain] = dict(stats)
    print(f"done {domain}: {stats['rows_all_time']:,} rows scanned, {stats['rows_2020_2022']:,} in-window rows in {stats['scan_seconds']} sec")

review_summary_rows = []
for domain, stats in review_summary.items():
    rows_window = stats.get('rows_2020_2022', 0)
    review_summary_rows.append({
        'domain': domain,
        'rows_all_time': stats.get('rows_all_time', 0),
        'rows_2020_2022': rows_window,
        'unique_users_2020_2022': len(review_user_counts_by_domain[domain]),
        'unique_asin_2020_2022': len(review_asin_sets_by_domain[domain]),
        'unique_parent_asin_2020_2022': len(review_parent_sets[domain]),
        'missing_parent_asin_pct_2020_2022': round(pct(stats.get('missing_parent_asin_2020_2022', 0), rows_window), 4),
        'empty_text_pct_2020_2022': round(pct(stats.get('empty_text_2020_2022', 0), rows_window), 4),
        'review_image_pct_2020_2022': round(pct(stats.get('review_rows_with_image_2020_2022', 0), rows_window), 4),
        'parents_missing_metadata_rows': review_parent_missing_metadata_counts[domain],
        'parents_missing_metadata_row_pct': round(pct(review_parent_missing_metadata_counts[domain], rows_window), 4),
        'scan_seconds': stats.get('scan_seconds', 0),
    })

show_table(
    review_summary_rows,
    title='Review shape and join-readiness summary by domain',
    note='This table uses only review rows in the 2020-2022 window for unique users/items and coverage metrics.',
)
write_csv(OUTPUT_DIR / 'eda2_review_shape_summary.csv', review_summary_rows, list(review_summary_rows[0].keys()))


Scanning reviews: Beauty_and_Personal_Care -> Beauty_and_Personal_Care.jsonl
done Beauty_and_Personal_Care: 23,911,390 rows scanned, 10,959,505 in-window rows in 99.6 sec

Scanning reviews: Clothing_Shoes_and_Jewelry -> Clothing_Shoes_and_Jewelry.jsonl
done Clothing_Shoes_and_Jewelry: 66,033,346 rows scanned, 29,078,609 in-window rows in 260.0 sec

Scanning reviews: Sports_and_Outdoors -> Sports_and_Outdoors.jsonl
done Sports_and_Outdoors: 19,595,170 rows scanned, 7,248,483 in-window rows in 76.0 sec

----------------------------------------------------------------------------------------------------
Review shape and join-readiness summary by domain
----------------------------------------------------------------------------------------------------
This table uses only review rows in the 2020-2022 window for unique users/items and coverage metrics.
Showing 3 of 3 rows


,domain,rows_all_time,rows_2020_2022,unique_users_2020_2022,unique_asin_2020_2022,unique_parent_asin_2020_2022,missing_parent_asin_pct_2020_2022,empty_text_pct_2020_2022,review_image_pct_2020_2022,parents_missing_metadata_rows,parents_missing_metadata_row_pct,scan_seconds
0,Beauty_and_Personal_Care,23911390,10959505,6155154,844360,596172,0.0,0.2512,9.9014,0,0.0,99.6
1,Clothing_Shoes_and_Jewelry,66033346,29078609,12819316,8277638,3096426,0.0,0.2653,7.8107,0,0.0,260.0
2,Sports_and_Outdoors,19595170,7248483,4806512,1091805,691739,0.0,0.1913,8.8724,0,0.0,76.0


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda2_review_shape_summary.csv (3 rows)


## EDA 1 - ASIN Integrity Audit

In [6]:
# EDA 1: ASIN integrity audit
parent_multi_domain = [
    {'parent_asin': parent, 'review_domains': decode_domains(mask), 'domain_count': bin(mask).count('1')}
    for parent, mask in review_parent_domain_mask.items()
    if bin(mask).count('1') > 1
]
parent_multi_domain.sort(key=lambda x: (-x['domain_count'], x['parent_asin']))

asin_multi_parent = [
    {'asin': asin, 'parent_asin_count': len(parents), 'parent_asins': '; '.join(sorted(parents)[:10])}
    for asin, parents in asin_to_parents.items()
    if len(parents) > 1
]
asin_multi_parent.sort(key=lambda x: (-x['parent_asin_count'], x['asin']))

metadata_duplicate_rows = []
for domain, counts in meta_parent_counts.items():
    for parent, count in counts.items():
        if count > 1:
            metadata_duplicate_rows.append({'domain': domain, 'parent_asin': parent, 'metadata_rows': count})
metadata_duplicate_rows.sort(key=lambda x: (-x['metadata_rows'], x['domain'], x['parent_asin']))

review_metadata_coverage_rows = []
for domain in REVIEW_FILES:
    review_parents = review_parent_sets[domain]
    meta_parents = meta_parent_sets[domain]
    missing_parents = review_parents - meta_parents
    review_metadata_coverage_rows.append({
        'domain': domain,
        'review_parent_asin_2020_2022': len(review_parents),
        'metadata_parent_asin': len(meta_parents),
        'review_parent_missing_metadata': len(missing_parents),
        'review_parent_missing_metadata_pct': round(pct(len(missing_parents), len(review_parents)), 4),
    })

eda1_summary_rows = [
    {'check': 'parent_asin in 2+ review domains', 'count': len(parent_multi_domain)},
    {'check': 'asin maps to 2+ parent_asin', 'count': len(asin_multi_parent)},
    {'check': 'metadata duplicate parent_asin rows', 'count': len(metadata_duplicate_rows)},
]
for domain, stats in review_summary.items():
    eda1_summary_rows.append({
        'check': f'{domain}: review rows missing parent_asin in 2020-2022',
        'count': stats.get('missing_parent_asin_2020_2022', 0),
    })

print_section(
    'EDA 1 - ASIN integrity audit',
    'Goal: confirm that item identifiers are stable enough for parent_asin-level recommendation and metadata joins.',
)
print_bullets('Key results to inspect', [
    f'Parent ASINs appearing in multiple review domains: {len(parent_multi_domain):,}',
    f'Child ASINs mapping to multiple parent_asin values: {len(asin_multi_parent):,}',
    f'Duplicate parent_asin rows inside metadata files: {len(metadata_duplicate_rows):,}',
    'For each domain, check missing parent_asin rows and review parent_asin coverage in metadata.',
])
show_table(
    eda1_summary_rows,
    title='Integrity check counts',
    note='High counts here indicate identifier or join risks that should be understood before modeling.',
)
show_table(
    review_metadata_coverage_rows,
    title='Review parent_asin to metadata coverage',
    note='Low missing-metadata percentages mean most reviewed items can receive text/image metadata features.',
)
show_table(
    parent_multi_domain[:20],
    title='Example parent_asin values in multiple review domains',
    note='If this table is empty, no cross-domain duplicate product families were found in the scanned data.',
)
show_table(
    asin_multi_parent[:20],
    title='Example asin values mapping to multiple parent_asin values',
    note='If this table is empty, child-to-parent ASIN mappings look stable in the scanned data.',
)

write_csv(OUTPUT_DIR / 'eda1_parent_asin_multi_review_domain.csv', parent_multi_domain, ['parent_asin', 'review_domains', 'domain_count'])
write_csv(OUTPUT_DIR / 'eda1_asin_multi_parent_asin.csv', asin_multi_parent, ['asin', 'parent_asin_count', 'parent_asins'])
write_csv(OUTPUT_DIR / 'eda1_metadata_duplicate_parent_asin.csv', metadata_duplicate_rows, ['domain', 'parent_asin', 'metadata_rows'])
write_csv(OUTPUT_DIR / 'eda1_review_metadata_coverage.csv', review_metadata_coverage_rows, list(review_metadata_coverage_rows[0].keys()))


EDA 1 - ASIN integrity audit
Goal: confirm that item identifiers are stable enough for parent_asin-level recommendation and metadata joins.

----------------------------------------------------------------------------------------------------
Key results to inspect
----------------------------------------------------------------------------------------------------
- Parent ASINs appearing in multiple review domains: 0
- Child ASINs mapping to multiple parent_asin values: 0
- Duplicate parent_asin rows inside metadata files: 0
- For each domain, check missing parent_asin rows and review parent_asin coverage in metadata.

----------------------------------------------------------------------------------------------------
Integrity check counts
----------------------------------------------------------------------------------------------------
High counts here indicate identifier or join risks that should be understood before modeling.
Showing 6 of 6 rows


,check,count
0,parent_asin in 2+ review domains,0
1,asin maps to 2+ parent_asin,0
2,metadata duplicate parent_asin rows,0
3,Beauty_and_Personal_Care: review rows missing parent_asin in 2020-2022,0
4,Clothing_Shoes_and_Jewelry: review rows missing parent_asin in 2020-2022,0
5,Sports_and_Outdoors: review rows missing parent_asin in 2020-2022,0



----------------------------------------------------------------------------------------------------
Review parent_asin to metadata coverage
----------------------------------------------------------------------------------------------------
Low missing-metadata percentages mean most reviewed items can receive text/image metadata features.
Showing 3 of 3 rows


,domain,review_parent_asin_2020_2022,metadata_parent_asin,review_parent_missing_metadata,review_parent_missing_metadata_pct
0,Beauty_and_Personal_Care,596172,1028914,0,0.0
1,Clothing_Shoes_and_Jewelry,3096426,7218481,0,0.0
2,Sports_and_Outdoors,691739,1587421,0,0.0



----------------------------------------------------------------------------------------------------
Example parent_asin values in multiple review domains
----------------------------------------------------------------------------------------------------
If this table is empty, no cross-domain duplicate product families were found in the scanned data.
(no rows)

----------------------------------------------------------------------------------------------------
Example asin values mapping to multiple parent_asin values
----------------------------------------------------------------------------------------------------
If this table is empty, child-to-parent ASIN mappings look stable in the scanned data.
(no rows)
CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda1_parent_asin_multi_review_domain.csv (0 rows)
CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda1_asin_multi_parent_asin.csv (0 rows)
CSV saved: /Users/frankwang1224/Projects/rcd_sys

## EDA 2 - Dataset Shape And Sparsity

In [7]:
# EDA 2: Dataset shape and sparsity
sparsity_rows = []
for domain in REVIEW_FILES:
    user_summary = distribution_summary(review_user_counts_by_domain[domain].values())
    parent_summary = distribution_summary(review_parent_counts_by_domain[domain].values())
    row = {
        'domain': domain,
        'rows_2020_2022': review_summary[domain].get('rows_2020_2022', 0),
        'unique_users': len(review_user_counts_by_domain[domain]),
        'unique_asin': len(review_asin_sets_by_domain[domain]),
        'unique_parent_asin': len(review_parent_sets[domain]),
        'user_interactions_mean': user_summary['mean'],
        'user_interactions_median': user_summary['median'],
        'user_interactions_p90': user_summary['p90'],
        'user_interactions_p99': user_summary['p99'],
        'parent_reviews_mean': parent_summary['mean'],
        'parent_reviews_median': parent_summary['median'],
        'parent_reviews_p90': parent_summary['p90'],
        'parent_reviews_p99': parent_summary['p99'],
        'parent_reviews_max': parent_summary['max'],
    }
    rating_int_counts = Counter()
    for rating, count in review_rating_counts_by_domain[domain].items():
        try:
            star = int(float(rating))
        except (TypeError, ValueError):
            continue
        if 1 <= star <= 5:
            rating_int_counts[star] += count
    for star in range(1, 6):
        row[f'rating_{star}_count'] = rating_int_counts[star]
    sparsity_rows.append(row)

print_section(
    'EDA 2 - Dataset shape and sparsity',
    'Goal: understand whether there is enough repeated user and item behavior for recommender modeling.',
)
print_bullets('Domain-level reading guide', [
    'rows_2020_2022 shows the usable interaction volume after the project date filter.',
    'user_interactions_median near 1 means many users have only one observed action in that domain.',
    'parent_reviews_median and parent_reviews_p90 show whether item coverage is broad or concentrated.',
    'The rating columns show whether the task is better treated as implicit feedback or explicit rating prediction.',
])
show_table(
    sparsity_rows,
    title='Sparsity summary by domain',
    note='Use unique_parent_asin as the main item count if modeling at product-family level.',
)
write_csv(OUTPUT_DIR / 'eda2_sparsity_distribution_summary.csv', sparsity_rows, list(sparsity_rows[0].keys()))

rating_rows = []
for domain, counts in review_rating_counts_by_domain.items():
    rating_int_counts = Counter()
    for rating, count in counts.items():
        try:
            star = int(float(rating))
        except (TypeError, ValueError):
            continue
        if 1 <= star <= 5:
            rating_int_counts[star] += count
    total = sum(rating_int_counts.values())
    for star in range(1, 6):
        count = rating_int_counts[star]
        rating_rows.append({'domain': domain, 'rating': star, 'count': count, 'pct': round(pct(count, total), 4)})

show_table(
    rating_rows,
    max_rows=30,
    title='Rating distribution by domain',
    note='A very high 4/5-star share supports treating reviews as positive implicit interactions.',
)
write_csv(OUTPUT_DIR / 'eda2_rating_distribution.csv', rating_rows, ['domain', 'rating', 'count', 'pct'])


EDA 2 - Dataset shape and sparsity
Goal: understand whether there is enough repeated user and item behavior for recommender modeling.

----------------------------------------------------------------------------------------------------
Domain-level reading guide
----------------------------------------------------------------------------------------------------
- rows_2020_2022 shows the usable interaction volume after the project date filter.
- user_interactions_median near 1 means many users have only one observed action in that domain.
- parent_reviews_median and parent_reviews_p90 show whether item coverage is broad or concentrated.
- The rating columns show whether the task is better treated as implicit feedback or explicit rating prediction.

----------------------------------------------------------------------------------------------------
Sparsity summary by domain
----------------------------------------------------------------------------------------------------
Use unique_

,domain,rows_2020_2022,unique_users,unique_asin,unique_parent_asin,user_interactions_mean,user_interactions_median,user_interactions_p90,user_interactions_p99,parent_reviews_mean,parent_reviews_median,parent_reviews_p90,parent_reviews_p99,parent_reviews_max,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count
0,Beauty_and_Personal_Care,10959505,6155154,844360,596172,1.781,1,3,10,18.383,3,27,276,26105,1545280,611330,779362,1047743,6975790
1,Clothing_Shoes_and_Jewelry,29078609,12819316,8277638,3096426,2.268,1,4,14,9.391,1,10,142,207693,3072719,1706642,2443207,3380725,18475316
2,Sports_and_Outdoors,7248483,4806512,1091805,691739,1.508,1,2,7,10.479,2,16,145,10648,867146,379106,490454,780933,4730844


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda2_sparsity_distribution_summary.csv (3 rows)

----------------------------------------------------------------------------------------------------
Rating distribution by domain
----------------------------------------------------------------------------------------------------
A very high 4/5-star share supports treating reviews as positive implicit interactions.
Showing 15 of 15 rows


,domain,rating,count,pct
0,Beauty_and_Personal_Care,1,1545280,14.0999
1,Beauty_and_Personal_Care,2,611330,5.5781
2,Beauty_and_Personal_Care,3,779362,7.1113
3,Beauty_and_Personal_Care,4,1047743,9.5601
4,Beauty_and_Personal_Care,5,6975790,63.6506
5,Clothing_Shoes_and_Jewelry,1,3072719,10.5669
6,Clothing_Shoes_and_Jewelry,2,1706642,5.8691
7,Clothing_Shoes_and_Jewelry,3,2443207,8.4021
8,Clothing_Shoes_and_Jewelry,4,3380725,11.6262
9,Clothing_Shoes_and_Jewelry,5,18475316,63.5358


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda2_rating_distribution.csv (15 rows)


## EDA 3 - Cross-Domain Transfer Viability

This section does a second targeted pass over Clothing/Sports and Beauty reviews to check whether source-domain reviews occurred before Beauty events, avoiding future-information leakage.

In [8]:
# EDA 3: Cross-domain transfer viability
mask_counts = Counter(user_domain_mask.values())
shared_user_rows = [
    {'user_group': 'Beauty only', 'count': mask_counts[1]},
    {'user_group': 'Clothing only', 'count': mask_counts[2]},
    {'user_group': 'Sports only', 'count': mask_counts[4]},
    {'user_group': 'Beauty + Clothing', 'count': mask_counts[1 | 2]},
    {'user_group': 'Beauty + Sports', 'count': mask_counts[1 | 4]},
    {'user_group': 'Clothing + Sports', 'count': mask_counts[2 | 4]},
    {'user_group': 'Beauty + Clothing + Sports', 'count': mask_counts[1 | 2 | 4]},
]

print_section(
    'EDA 3 - Cross-domain transfer viability',
    'Goal: test whether Clothing/Sports behavior can help predict Beauty preferences without time leakage.',
)
show_table(
    shared_user_rows,
    title='Shared users across review domains',
    note='Cross-domain transfer is only useful when Beauty users also appear in Clothing and/or Sports.',
)
write_csv(OUTPUT_DIR / 'eda3_shared_users_by_domain_combo.csv', shared_user_rows, ['user_group', 'count'])

# Event-level non-leakage check: collect Clothing/Sports histories only for users who appear in Beauty.
source_history_by_user = defaultdict(list)
source_rating_sum_by_user = defaultdict(float)
source_rating_count_by_user = defaultdict(int)
source_parent_by_user = defaultdict(Counter)
source_domain_count_by_user = defaultdict(Counter)

for domain in ['Clothing_Shoes_and_Jewelry', 'Sports_and_Outdoors']:
    path = REVIEW_FILES[domain]
    print(f'\nCollecting source-domain histories for Beauty users: {domain}')
    t0 = time.time()
    kept = 0
    for row in iter_jsonl(path, MAX_REVIEW_ROWS_PER_FILE):
        if not in_window(row.get('timestamp')):
            continue
        user = safe_id(row.get('user_id'))
        if user not in beauty_users:
            continue
        ts_ms = to_ms(row.get('timestamp'))
        if ts_ms is None:
            continue
        kept += 1
        parent = safe_id(row.get('parent_asin'))
        rating = row.get('rating')
        source_history_by_user[user].append((ts_ms, domain, parent, rating))
        source_domain_count_by_user[user][domain] += 1
        if parent:
            source_parent_by_user[user][parent] += 1
        if rating is not None:
            try:
                source_rating_sum_by_user[user] += float(rating)
                source_rating_count_by_user[user] += 1
            except (TypeError, ValueError):
                pass
    print(f'kept {kept:,} source reviews for Beauty users from {domain} in {round(time.time() - t0, 1)} sec')

for user, events in source_history_by_user.items():
    events.sort(key=lambda x: x[0])

source_timestamps_by_user = {
    user: [event[0] for event in events]
    for user, events in source_history_by_user.items()
}
source_domains_by_user = {
    user: [event[1] for event in events]
    for user, events in source_history_by_user.items()
}

beauty_event_count = 0
beauty_events_with_prior_source = 0
beauty_events_with_prior_clothing = 0
beauty_events_with_prior_sports = 0
prior_source_counts_per_beauty_event = []
beauty_users_with_prior_source = set()

print('\nChecking prior source histories before each Beauty event')
for row in iter_jsonl(REVIEW_FILES['Beauty_and_Personal_Care'], MAX_REVIEW_ROWS_PER_FILE):
    if not in_window(row.get('timestamp')):
        continue
    user = safe_id(row.get('user_id'))
    ts_ms = to_ms(row.get('timestamp'))
    if not user or ts_ms is None:
        continue
    beauty_event_count += 1
    timestamps = source_timestamps_by_user.get(user, [])
    if not timestamps:
        prior_source_counts_per_beauty_event.append(0)
        continue
    prior_n = bisect_left(timestamps, ts_ms)
    prior_source_counts_per_beauty_event.append(prior_n)
    if prior_n > 0:
        beauty_events_with_prior_source += 1
        beauty_users_with_prior_source.add(user)
        prior_domains = set(source_domains_by_user[user][:prior_n])
        if 'Clothing_Shoes_and_Jewelry' in prior_domains:
            beauty_events_with_prior_clothing += 1
        if 'Sports_and_Outdoors' in prior_domains:
            beauty_events_with_prior_sports += 1

prior_source_summary = distribution_summary(prior_source_counts_per_beauty_event)
eda3_prior_rows = [{
    'beauty_events_2020_2022': beauty_event_count,
    'beauty_events_with_any_prior_clothing_or_sports': beauty_events_with_prior_source,
    'pct_beauty_events_with_prior_source': round(pct(beauty_events_with_prior_source, beauty_event_count), 4),
    'beauty_events_with_prior_clothing': beauty_events_with_prior_clothing,
    'beauty_events_with_prior_sports': beauty_events_with_prior_sports,
    'beauty_users_2020_2022': len(beauty_users),
    'beauty_users_with_prior_source': len(beauty_users_with_prior_source),
    'pct_beauty_users_with_prior_source': round(pct(len(beauty_users_with_prior_source), len(beauty_users)), 4),
    'prior_source_count_median_per_beauty_event': prior_source_summary['median'],
    'prior_source_count_p90_per_beauty_event': prior_source_summary['p90'],
    'prior_source_count_p99_per_beauty_event': prior_source_summary['p99'],
    'prior_source_count_max_per_beauty_event': prior_source_summary['max'],
}]

show_table(
    eda3_prior_rows,
    title='Prior Clothing/Sports history before Beauty events',
    note='This is the strongest leakage-safe signal: source-domain behavior must occur before the Beauty event.',
)
write_csv(OUTPUT_DIR / 'eda3_prior_source_history_before_beauty.csv', eda3_prior_rows, list(eda3_prior_rows[0].keys()))

rating_tendency_rows = []
for user in beauty_users:
    if beauty_user_rating_count[user] == 0 or source_rating_count_by_user[user] == 0:
        continue
    beauty_avg = beauty_user_rating_sum[user] / beauty_user_rating_count[user]
    source_avg = source_rating_sum_by_user[user] / source_rating_count_by_user[user]
    rating_tendency_rows.append({
        'user_id': user,
        'beauty_avg_rating': round(beauty_avg, 4),
        'source_avg_rating': round(source_avg, 4),
        'source_minus_beauty_avg': round(source_avg - beauty_avg, 4),
        'beauty_rating_count': beauty_user_rating_count[user],
        'source_rating_count': source_rating_count_by_user[user],
    })

rating_diff_summary = distribution_summary([round((r['source_minus_beauty_avg']) * 1000) for r in rating_tendency_rows])
rating_tendency_summary = [{
    'users_with_beauty_and_source_ratings': len(rating_tendency_rows),
    'source_minus_beauty_avg_x1000_mean': rating_diff_summary['mean'],
    'source_minus_beauty_avg_x1000_median': rating_diff_summary['median'],
    'source_minus_beauty_avg_x1000_p90': rating_diff_summary['p90'],
    'source_minus_beauty_avg_x1000_p99': rating_diff_summary['p99'],
}]

show_table(
    rating_tendency_summary,
    title='Cross-domain rating tendency summary',
    note='This compares users average source-domain ratings against their Beauty ratings.',
)
write_csv(OUTPUT_DIR / 'eda3_rating_tendency_summary.csv', rating_tendency_summary, list(rating_tendency_summary[0].keys()))
write_csv(OUTPUT_DIR / 'eda3_rating_tendency_user_level_examples.csv', rating_tendency_rows[:5000], list(rating_tendency_rows[0].keys()) if rating_tendency_rows else ['user_id'])

# Category path similarity for source items touched by Beauty users.
source_parent_counts_for_beauty_users = Counter()
for user_counter in source_parent_by_user.values():
    source_parent_counts_for_beauty_users.update(user_counter)

source_category_rows = []
for parent, count in source_parent_counts_for_beauty_users.most_common(20000):
    paths = meta_category_path_by_parent.get(parent)
    if not paths:
        continue
    path_value, path_count = paths.most_common(1)[0]
    source_category_rows.append({
        'category_path': path_value,
        'source_review_count_from_beauty_users': count,
        'example_parent_asin': parent,
    })

category_path_counts = Counter()
for row in source_category_rows:
    category_path_counts[row['category_path']] += row['source_review_count_from_beauty_users']
category_path_summary_rows = [
    {'category_path': path, 'source_review_count_from_beauty_users': count}
    for path, count in category_path_counts.most_common(50)
]

show_table(
    category_path_summary_rows,
    max_rows=30,
    title='Top source-domain category paths among items reviewed by Beauty users',
    note='This helps explain what kind of Clothing/Sports behavior may transfer into Beauty recommendation.',
)
write_csv(OUTPUT_DIR / 'eda3_source_category_paths_for_beauty_users.csv', category_path_summary_rows, ['category_path', 'source_review_count_from_beauty_users'])


EDA 3 - Cross-domain transfer viability
Goal: test whether Clothing/Sports behavior can help predict Beauty preferences without time leakage.

----------------------------------------------------------------------------------------------------
Shared users across review domains
----------------------------------------------------------------------------------------------------
Cross-domain transfer is only useful when Beauty users also appear in Clothing and/or Sports.
Showing 7 of 7 rows


,user_group,count
0,Beauty only,2539588
1,Clothing only,8039576
2,Sports only,2086367
3,Beauty + Clothing,2363304
4,Beauty + Sports,303709
5,Clothing + Sports,1467883
6,Beauty + Clothing + Sports,948553


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda3_shared_users_by_domain_combo.csv (7 rows)

kept 12,214,898 source reviews for Beauty users from Clothing_Shoes_and_Jewelry in 165.5 sec

kept 2,340,455 source reviews for Beauty users from Sports_and_Outdoors in 45.6 sec

Checking prior source histories before each Beauty event

----------------------------------------------------------------------------------------------------
Prior Clothing/Sports history before Beauty events
----------------------------------------------------------------------------------------------------
This is the strongest leakage-safe signal: source-domain behavior must occur before the Beauty event.
Showing 1 of 1 rows


,beauty_events_2020_2022,beauty_events_with_any_prior_clothing_or_sports,pct_beauty_events_with_prior_source,beauty_events_with_prior_clothing,beauty_events_with_prior_sports,beauty_users_2020_2022,beauty_users_with_prior_source,pct_beauty_users_with_prior_source,prior_source_count_median_per_beauty_event,prior_source_count_p90_per_beauty_event,prior_source_count_p99_per_beauty_event,prior_source_count_max_per_beauty_event
0,10959505,5632693,51.3955,5213078,2069048,6155154,2666289,43.318,1,7,79,3198


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda3_prior_source_history_before_beauty.csv (1 rows)

----------------------------------------------------------------------------------------------------
Cross-domain rating tendency summary
----------------------------------------------------------------------------------------------------
This compares users average source-domain ratings against their Beauty ratings.
Showing 1 of 1 rows


,users_with_beauty_and_source_ratings,source_minus_beauty_avg_x1000_mean,source_minus_beauty_avg_x1000_median,source_minus_beauty_avg_x1000_p90,source_minus_beauty_avg_x1000_p99
0,3615566,85.165,0,2000,4000


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda3_rating_tendency_summary.csv (1 rows)
CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda3_rating_tendency_user_level_examples.csv (5,000 rows)

----------------------------------------------------------------------------------------------------
Top source-domain category paths among items reviewed by Beauty users
----------------------------------------------------------------------------------------------------
This helps explain what kind of Clothing/Sports behavior may transfer into Beauty recommendation.
Showing 30 of 50 rows


,category_path,source_review_count_from_beauty_users
0,"Clothing, Shoes & Jewelry > Women > Clothing > Lingerie, Sleep & Lounge > Li...",113282
1,"Clothing, Shoes & Jewelry > Novelty & More > Clothing > Novelty > Women > To...",109037
2,"Clothing, Shoes & Jewelry > Women > Clothing > Tops, Tees & Blouses > Tunics",104367
3,"Clothing, Shoes & Jewelry > Women > Shoes > Slippers",102987
4,"Clothing, Shoes & Jewelry > Women > Clothing > Dresses > Casual",96144
5,"Clothing, Shoes & Jewelry > Women > Clothing > Dresses",94416
6,"Clothing, Shoes & Jewelry > Women > Clothing > Tops, Tees & Blouses > T-Shirts",91270
7,"Clothing, Shoes & Jewelry > Women > Clothing > Tops, Tees & Blouses > Blouse...",83197
8,Sports & Outdoors > Sports & Outdoor Recreation Accessories > Sports Water B...,64516
9,"Clothing, Shoes & Jewelry > Women > Clothing > Lingerie, Sleep & Lounge > Li...",64295


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda3_source_category_paths_for_beauty_users.csv (50 rows)


## EDA 4 - Metadata/Text/Image Readiness

In [10]:
# EDA 4: Metadata/text/image readiness plus review image coverage
eda4_rows = []
for domain, stats in meta_summary.items():
    rows = stats['rows']
    review_rows = review_summary[domain].get('rows_2020_2022', 0)
    eda4_rows.append({
        'domain': domain,
        'metadata_rows': rows,
        'empty_title_pct': round(pct(stats.get('empty_title', 0), rows), 4),
        'empty_features_pct': round(pct(stats.get('empty_features', 0), rows), 4),
        'empty_description_pct': round(pct(stats.get('empty_description', 0), rows), 4),
        'embedding_text_pct': round(pct(stats.get('rows_with_embedding_text', 0), rows), 4),
        'text_len_median': stats.get('text_len_median', 0),
        'text_len_p90': stats.get('text_len_p90', 0),
        'text_len_p99': stats.get('text_len_p99', 0),
        'metadata_usable_image_pct': round(pct(stats.get('rows_with_usable_image', 0), rows), 4),
        'review_image_pct_2020_2022': round(pct(review_image_rows_by_domain[domain], review_rows), 4),
        'empty_price_pct': round(pct(stats.get('empty_price', 0), rows), 4),
    })

print_section(
    'EDA 4 - Metadata/text/image readiness',
    'Goal: decide whether metadata text, metadata images, review images, and price are usable as side features.',
)
print_bullets('Feature-readiness reading guide', [
    'embedding_text_pct is the percentage of metadata rows with non-empty title/features/description text.',
    'metadata_usable_image_pct is the main readiness check for CLIP-style product image embeddings.',
    'review_image_pct_2020_2022 is usually much lower; treat user-uploaded review images as optional.',
    'empty_price_pct tells whether price should be a core feature or only an optional feature.',
])
show_table(
    eda4_rows,
    title='Text, image, review-image, and price readiness by domain',
    note='Use this table before committing to BGE text embeddings or CLIP image embeddings.',
)
write_csv(OUTPUT_DIR / 'eda4_text_image_price_readiness.csv', eda4_rows, list(eda4_rows[0].keys()))

main_category_rows = []
for domain, counts in meta_main_category_counts.items():
    total = sum(counts.values())
    for value, count in counts.most_common(20):
        main_category_rows.append({'domain': domain, 'main_category': value, 'count': count, 'pct': round(pct(count, total), 4)})

first_category_rows = []
for domain, counts in meta_first_category_counts.items():
    total = sum(counts.values())
    for value, count in counts.most_common(20):
        first_category_rows.append({'domain': domain, 'first_categories_value': value, 'count': count, 'pct': round(pct(count, total), 4)})

show_table(
    main_category_rows,
    max_rows=60,
    title='Top metadata main_category values',
    note='This field can be null or mixed, so it should not be the first source of domain truth.',
)
show_table(
    first_category_rows,
    max_rows=60,
    title='Top metadata categories[0] values',
    note='This field is often cleaner than main_category, but still needs auditing.',
)

write_csv(OUTPUT_DIR / 'eda4_main_category_distribution.csv', main_category_rows, ['domain', 'main_category', 'count', 'pct'])
write_csv(OUTPUT_DIR / 'eda4_first_category_distribution.csv', first_category_rows, ['domain', 'first_categories_value', 'count', 'pct'])


EDA 4 - Metadata/text/image readiness
Goal: decide whether metadata text, metadata images, review images, and price are usable as side features.

----------------------------------------------------------------------------------------------------
Feature-readiness reading guide
----------------------------------------------------------------------------------------------------
- embedding_text_pct is the percentage of metadata rows with non-empty title/features/description text.
- metadata_usable_image_pct is the main readiness check for CLIP-style product image embeddings.
- review_image_pct_2020_2022 is usually much lower; treat user-uploaded review images as optional.
- empty_price_pct tells whether price should be a core feature or only an optional feature.

----------------------------------------------------------------------------------------------------
Text, image, review-image, and price readiness by domain
--------------------------------------------------------------------

,domain,metadata_rows,empty_title_pct,empty_features_pct,empty_description_pct,embedding_text_pct,text_len_median,text_len_p90,text_len_p99,metadata_usable_image_pct,review_image_pct_2020_2022,empty_price_pct
0,Beauty_and_Personal_Care,1028914,0.0052,27.5373,45.1489,99.9989,627,2022,3312,99.9922,9.9014,63.0018
1,Clothing_Shoes_and_Jewelry,7218481,0.0082,10.3165,47.4484,99.9993,541,1698,3476,99.8608,7.8107,79.5460
2,Sports_and_Outdoors,1587421,0.0071,16.9191,34.8365,99.9988,591,1864,3448,99.9660,8.8724,69.4262


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda4_text_image_price_readiness.csv (3 rows)

----------------------------------------------------------------------------------------------------
Top metadata main_category values
----------------------------------------------------------------------------------------------------
This field can be null or mixed, so it should not be the first source of domain truth.
Showing 60 of 60 rows


,domain,main_category,count,pct
0,Beauty_and_Personal_Care,All Beauty,737549,71.6823
1,Beauty_and_Personal_Care,(null),102896,10.0004
2,Beauty_and_Personal_Care,Health & Personal Care,71800,6.9782
3,Beauty_and_Personal_Care,Premium Beauty,34898,3.3917
4,Beauty_and_Personal_Care,Amazon Home,32143,3.1240
5,Beauty_and_Personal_Care,Tools & Home Improvement,14387,1.3983
6,Beauty_and_Personal_Care,AMAZON FASHION,13055,1.2688
7,Beauty_and_Personal_Care,Industrial & Scientific,4901,0.4763
8,Beauty_and_Personal_Care,Sports & Outdoors,2425,0.2357
9,Beauty_and_Personal_Care,Grocery,2181,0.2120



----------------------------------------------------------------------------------------------------
Top metadata categories[0] values
----------------------------------------------------------------------------------------------------
This field is often cleaner than main_category, but still needs auditing.
Showing 6 of 6 rows


,domain,first_categories_value,count,pct
0,Beauty_and_Personal_Care,Beauty & Personal Care,1028914,100.0000
1,Clothing_Shoes_and_Jewelry,"Clothing, Shoes & Jewelry",7217166,99.9818
2,Clothing_Shoes_and_Jewelry,"Shoe, Jewelry & Watch Accessories",1315,0.0182
3,Sports_and_Outdoors,Sports & Outdoors,1496313,99.9147
4,Sports_and_Outdoors,Sports & Outdoor Recreation Accessories,803,0.0536
5,Sports_and_Outdoors,Hunting & Fishing,475,0.0317


CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda4_main_category_distribution.csv (60 rows)
CSV saved: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda4_first_category_distribution.csv (6 rows)


In [11]:
print_section('EDA complete')
print('Output files written under:', OUTPUT_DIR)
print('\nSuggested interpretation order:')
print('1. Read eda1_review_metadata_coverage.csv and eda1_* anomaly files first.')
print('2. Use eda2_* to decide parent_asin vs asin modeling and whether sparsity is severe.')
print('3. Use eda3_* to decide whether Clothing/Sports can support Beauty recommendation without time leakage.')
print('4. Use eda4_* to decide text/image feature coverage before BGE/CLIP embedding work.')
print_output_inventory(OUTPUT_DIR)

gc.collect()


EDA complete
Output files written under: /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs

Suggested interpretation order:
1. Read eda1_review_metadata_coverage.csv and eda1_* anomaly files first.
2. Use eda2_* to decide parent_asin vs asin modeling and whether sparsity is severe.
3. Use eda3_* to decide whether Clothing/Sports can support Beauty recommendation without time leakage.
4. Use eda4_* to decide text/image feature coverage before BGE/CLIP embedding work.

Output CSV inventory
These are the files produced by the notebook run.
- eda1_asin_multi_parent_asin.csv (0.0 KB) -> /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda1_asin_multi_parent_asin.csv
- eda1_metadata_duplicate_parent_asin.csv (0.0 KB) -> /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda1_metadata_duplicate_parent_asin.csv
- eda1_parent_asin_multi_review_domain.csv (0.0 KB) -> /Users/frankwang1224/Projects/rcd_sys_proj02/eda_outputs/eda1_parent_asin_multi_review_domain.csv
- eda1_rev

0